# Lab 16: VEGAS tutorial

We have used random samples to measure averages in the Ising model and in a path integral. Today we will choose where to sample so an integral can be measured more efficiently.

We will also use those samples to generate events with a specified distribution.

PHYS 142/242 · Thursday, November 19, 2026

## Run the notebook

Run all cells once in notebook view, then enter RISE. The setup cells define the samplers and plotting helpers. Each calculation uses a stated seed.

During the presentation, rerun a calculation to reproduce its result. The source notebook is saved with empty outputs.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display

plt.rcParams.update({'font.size': 12, 'figure.figsize': (8.4, 3.5),
                     'figure.dpi': 110, 'figure.constrained_layout.use': True})

In [ ]:
A1, A2 = 1.0, 4.0
MU1X = MU1Y = MU2X = MU2Y = 2.5
SIGMA1X = SIGMA1Y = 1.0
SIGMA2X = SIGMA2Y = 0.1
XMIN = YMIN = 0.0
XMAX = YMAX = 5.0

In [ ]:
def components():
    return ((A1, MU1X, MU1Y, SIGMA1X, SIGMA1Y),
            (A2, MU2X, MU2Y, SIGMA2X, SIGMA2Y))

def f(x, y):
    return sum(a * np.exp(-0.5 * (((x-mx)/sx)**2 + ((y-my)/sy)**2))
               for a, mx, my, sx, sy in components())

def exact_integral():
    """Analytic finite-box integral of the two unnormalized Gaussians."""
    total = 0.0
    for a, mx, my, sx, sy in components():
        ix = sx * math.sqrt(math.pi/2) * (
            math.erf((XMAX-mx)/(math.sqrt(2)*sx))
            - math.erf((XMIN-mx)/(math.sqrt(2)*sx)))
        iy = sy * math.sqrt(math.pi/2) * (
            math.erf((YMAX-my)/(math.sqrt(2)*sy))
            - math.erf((YMIN-my)/(math.sqrt(2)*sy)))
        total += a * ix * iy
    return total

def draw_proposals(xLow, yLow, size, rng):
    """Choose each interval equally often, then sample within its own width."""
    dx, dy = np.diff(xLow), np.diff(yLow)
    ix = rng.integers(len(dx), size=size)
    iy = rng.integers(len(dy), size=size)
    x = xLow[ix] + dx[ix] * rng.random(size)
    y = yLow[iy] + dy[iy] * rng.random(size)
    q = 1.0 / (len(dx) * len(dy) * dx[ix] * dy[iy])
    return x, y, q

def rebin(edges, scores, alpha_damp):
    """Redistribute edges using the estimated importance of each interval."""
    z = scores / scores.sum()
    if np.any(z <= 0) or not np.all(np.isfinite(z)):
        raise ValueError('Adaptation requires finite positive bin scores.')
    if alpha_damp is not None:
        z = ((z - 1) / np.log(z)) ** alpha_damp
    cumulative = np.r_[0.0, np.cumsum(z)]
    new = np.interp(np.linspace(0, cumulative[-1], len(edges)), cumulative, edges)
    if np.any(np.diff(new) <= 0):
        raise ValueError('Adaptation produced a collapsed interval.')
    return new

def setup_intervals(NN=40, KK=2000, nIterations=20, alpha_damp=1.5, seed=None):
    """Learn a product proposal; alpha_damp=None selects undamped allocation."""
    if NN < 2 or KK < NN or nIterations < 0:
        raise ValueError('Require NN >= 2, KK >= NN, and nIterations >= 0.')
    if not (XMIN < XMAX and YMIN < YMAX):
        raise ValueError('Each domain must have positive width.')
    if alpha_damp is not None and alpha_damp <= 0:
        raise ValueError('alpha_damp must be positive or None.')
    xLow = np.linspace(XMIN, XMAX, NN+1)
    yLow = np.linspace(YMIN, YMAX, NN+1)
    rng = np.random.default_rng(seed)
    for _ in range(nIterations):
        dx, dy = np.diff(xLow), np.diff(yLow)
        x, y, q = draw_proposals(xLow, yLow, KK, rng)
        ix = np.searchsorted(xLow, x, side='right') - 1
        iy = np.searchsorted(yLow, y, side='right') - 1
        nx, ny = np.bincount(ix, minlength=NN), np.bincount(iy, minlength=NN)
        if np.any(nx == 0) or np.any(ny == 0):
            raise ValueError('An adaptation bin is empty; increase KK.')
        values = f(x, y)
        px, py = 1/(NN*dx[ix]), 1/(NN*dy[iy])
        sx = dx * np.sqrt(np.bincount(ix, weights=(values/py)**2, minlength=NN)/nx)
        sy = dy * np.sqrt(np.bincount(iy, weights=(values/px)**2, minlength=NN)/ny)
        xLow, yLow = rebin(xLow, sx, alpha_damp), rebin(yLow, sy, alpha_damp)
    return xLow, yLow, np.diff(xLow), np.diff(yLow)

def rejection_bound(xLow, yLow):
    """Conservative global bound on f/q for this positive Gaussian mixture.

    Each component is largest at the point in a cell closest to its mean.
    The sum of those component maxima bounds the mixture throughout that cell.
    """
    upper = np.zeros((len(xLow)-1, len(yLow)-1))
    for a, mx, my, sx, sy in components():
        if a < 0 or sx <= 0 or sy <= 0:
            raise ValueError('Require nonnegative amplitudes and positive widths.')
        cx = np.clip(mx, xLow[:-1], xLow[1:])
        cy = np.clip(my, yLow[:-1], yLow[1:])
        upper += a*np.exp(-0.5*(((cx[:,None]-mx)/sx)**2 + ((cy[None,:]-my)/sy)**2))
    cell_inverse_q = (len(xLow)-1)*(len(yLow)-1)*np.outer(np.diff(xLow), np.diff(yLow))
    bound = float(np.max(upper * cell_inverse_q))
    if not np.isfinite(bound) or bound <= 0:
        raise ValueError('The rejection bound must be finite and positive.')
    return np.nextafter(bound, np.inf)

def sample_events(nPoints, xLow, yLow, seed=None):
    """Independent rejection sampling with a proved Gaussian-mixture envelope."""
    if nPoints < 1:
        raise ValueError('nPoints must be positive.')
    rng = np.random.default_rng(seed)
    bound = rejection_bound(xLow, yLow)
    accepted, proposals = [], []
    naccepted = neval = 0
    maximum = 0.0
    while naccepted < nPoints:
        x, y, q = draw_proposals(xLow, yLow, 4096, rng)
        weights = f(x, y)/q
        maximum = max(maximum, float(weights.max()))
        if np.any(weights > bound*(1+1e-12)):
            raise ValueError('Envelope violated; these samples are invalid.')
        keep = rng.random(len(x)) < weights/bound
        accepted.append(np.column_stack((x[keep], y[keep])))
        proposals.append(np.column_stack((x, y)))
        naccepted += int(keep.sum())
        neval += len(x)
    return {'events': np.concatenate(accepted)[:nPoints],
            'proposals': np.concatenate(proposals), 'nFunEval': neval,
            'nAccepted': naccepted, 'maxWeightEncountered': maximum, 'bound': bound}

def brute_force(nPoints, seed=None):
    result = sample_events(nPoints, np.array([XMIN, XMAX]), np.array([YMIN, YMAX]), seed)
    return {**result, 'yy1': result['events'][:,0], 'yy2': result['events'][:,1],
            'nTrainEval': 0, 'nTotalEval': result['nFunEval']}

def vegas(nPoints, NN=40, KK=2000, nIterations=20, alpha_damp=1.5, seed=None):
    train_seed, sample_seed = np.random.SeedSequence(seed).spawn(2)
    xLow, yLow, dx, dy = setup_intervals(NN, KK, nIterations, alpha_damp, train_seed)
    result = sample_events(nPoints, xLow, yLow, sample_seed)
    return {**result, 'yy1vegas': result['events'][:,0], 'yy2vegas': result['events'][:,1],
            'yy1vrho': result['proposals'][:,0], 'yy2vrho': result['proposals'][:,1],
            'vegasRatioMax': result['bound'], 'xLow': xLow, 'yLow': yLow,
            'nTrainEval': KK*nIterations, 'nTotalEval': KK*nIterations+result['nFunEval']}

def estimate_integral(xLow, yLow, nSamples, seed=None):
    """Use an independent fixed-size sample with the trained grid frozen."""
    if nSamples < 2:
        raise ValueError('At least two samples are required for an uncertainty.')
    x, y, q = draw_proposals(xLow, yLow, nSamples, np.random.default_rng(seed))
    weights = f(x,y)/q
    return {'mean': float(weights.mean()),
            'stderr': float(weights.std(ddof=1)/np.sqrt(nSamples)),
            'nFunEval': nSamples, 'weights': weights}

def lego_plot(xAmplitudes, yAmplitudes, nBins, xLabel, yLabel, title):
    hist, xe, ye = np.histogram2d(xAmplitudes, yAmplitudes, bins=nBins,
                                range=((XMIN,XMAX),(YMIN,YMAX)))
    xx, yy = np.meshgrid(xe[:-1], ye[:-1], indexing='ij')
    fig = plt.figure(figsize=(7,4.5))
    ax = fig.add_subplot(111, projection='3d')
    ax.bar3d(xx.ravel(), yy.ravel(), np.zeros(hist.size),
             np.diff(xe)[0], np.diff(ye)[0], hist.ravel(), color='#007c91')
    ax.set(xlabel=xLabel, ylabel=yLabel, zlabel='Events', title=title,
           xlim=(XMIN,XMAX), ylim=(YMIN,YMAX))
    plt.show()
    return ax

def plot_results(nPoints=100_000, nBins=40, NN=40, KK=2000,
                 nIterations=20, alpha_damp=1.5, seed=142):
    bf = brute_force(nPoints, seed)
    vg = vegas(nPoints, NN, KK, nIterations, alpha_damp, seed)
    lego_plot(bf['yy1'], bf['yy2'], nBins, 'x', 'y', 'Uniform rejection: accepted events')
    lego_plot(vg['yy1vrho'], vg['yy2vrho'], nBins, 'x', 'y', 'Adaptive proposals q(x,y)')
    lego_plot(vg['yy1vegas'], vg['yy2vegas'], nBins, 'x', 'y', 'Adaptive rejection: accepted events')
    for name, result in [('Uniform',bf), ('Adaptive',vg)]:
        print(f"{name}: training={result['nTrainEval']:,}; production={result['nFunEval']:,}; "
              f"total={result['nTotalEval']:,}; returned events={nPoints:,}")
    return bf, vg

In [ ]:
def visual():
    display(HTML('<span class="lab16-visual-marker"></span>'))


def target_picture():
    visual()
    grid = np.linspace(XMIN, XMAX, 301)
    yy = np.linspace(YMIN, YMAX, 301)
    xx, yy2 = np.meshgrid(grid, yy, indexing='ij')
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
    im = axes[0].imshow(f(xx,yy2).T, origin='lower', extent=(XMIN,XMAX,YMIN,YMAX),
                        aspect='auto', cmap='viridis')
    fig.colorbar(im, ax=axes[0], label='f(x,y)')
    axes[0].set(xlabel='x', ylabel='y', title='A broad bump and a narrow peak')
    axes[1].plot(grid, f(grid, MU1Y), color='#007c91', lw=2)
    axes[1].set(xlabel='x', ylabel='f(x, 2.5)', title='A slice through the center')
    plt.show()


def proposal_picture(xedges, yedges):
    visual()
    centers = (xedges[1:]+xedges[:-1])/2
    density = 1/(len(centers)*np.diff(xedges))
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
    axes[0].vlines(xedges, 0, 1, color='#007c91', lw=1)
    axes[0].set(xlim=(XMIN,XMAX), yticks=[], xlabel='x', title='Learned interval boundaries')
    axes[1].stairs(density, xedges, color='#007c91', lw=2, label='Learned q_x')
    axes[1].axhline(1/(XMAX-XMIN), color='#bc5c21', ls='--', label='Uniform')
    axes[1].set(xlabel='x', ylabel='Proposal density', title='Equal probability in unequal widths')
    axes[1].legend()
    plt.show()


def weight_picture(uniform, adaptive):
    visual()
    fig, ax = plt.subplots(figsize=(8.4,3.5))
    bins = np.linspace(0, max(uniform['weights'].max(),adaptive['weights'].max()), 65)
    ax.hist(uniform['weights'], bins=bins, density=True, histtype='step', lw=2,
            color='#bc5c21', label='Uniform proposal')
    ax.hist(adaptive['weights'], bins=bins, density=True, histtype='step', lw=2,
            color='#007c91', label='Adaptive proposal')
    ax.axvline(exact_integral(), color='#172b4d', ls=':', label='Exact integral')
    ax.set(xlabel='Importance weight f/q', ylabel='Density', title='The spread of the weights sets the error')
    ax.legend()
    plt.show()


def estimate_picture(uniform, adaptive):
    visual()
    fig, ax = plt.subplots(figsize=(8.4,3.5))
    for i,(name,result,color) in enumerate([('Uniform',uniform,'#bc5c21'),('Adaptive',adaptive,'#007c91')]):
        ax.errorbar(result['mean'],i,xerr=result['stderr'],fmt='o',color=color,capsize=5)
        ax.annotate(f"{result['mean']:.4f} ± {result['stderr']:.4f}",
                    (result['mean'],i),xytext=(0,16),textcoords='offset points',ha='center')
    ax.axvline(exact_integral(),color='#172b4d',ls='--',label=f'Exact: {exact_integral():.6f}')
    ax.set(yticks=[0,1],yticklabels=['Uniform','Adaptive'],ylim=(-.6,1.7),xlabel='Integral estimate')
    ax.legend(loc='lower right')
    plt.show()


def event_picture(bf, vg):
    visual()
    arrays=[bf['events'], vg['proposals'], vg['events']]
    titles=['Uniform: accepted events','Adaptive: proposals q','Adaptive: accepted events']
    fig, axes=plt.subplots(1,3,figsize=(10,3.1))
    for ax,points,title in zip(axes,arrays,titles):
        hist,xe,ye=np.histogram2d(points[:,0],points[:,1],bins=60,
                                range=((XMIN,XMAX),(YMIN,YMAX)),density=True)
        im=ax.imshow(hist.T,origin='lower',extent=(XMIN,XMAX,YMIN,YMAX),
                     aspect='auto',cmap='viridis',vmin=0,vmax=.85)
        ax.set(xlabel='x',ylabel='y',title=title)
    fig.colorbar(im, ax=axes, label='Probability density', shrink=.85)
    plt.show()


def cost_picture(bf, vg):
    visual()
    fig,ax=plt.subplots(figsize=(8.4,3.3))
    production=[bf['nFunEval'],vg['nFunEval']]
    training=[bf['nTrainEval'],vg['nTrainEval']]
    ax.barh(['Uniform','Adaptive'],production,color='#007c91',label='Production')
    ax.barh(['Uniform','Adaptive'],training,left=production,color='#bc5c21',label='Training')
    for i,total in enumerate([bf['nTotalEval'],vg['nTotalEval']]):
        ax.text(total+15000,i,f'{total:,}',va='center')
    ax.set(xlabel='Function evaluations',xlim=(0,max(bf['nTotalEval'],vg['nTotalEval'])*1.18))
    ax.legend()
    plt.show()


def damping_picture(damped_x, raw_x, damped_result, raw_result):
    visual()
    fig,axes=plt.subplots(1,2,figsize=(9,3.5))
    for edges,label,color in [(damped_x,'Damped','#007c91'),(raw_x,'Raw scores','#bc5c21')]:
        axes[0].stairs(1/((len(edges)-1)*np.diff(edges)),edges,label=label,color=color,lw=1.8)
    axes[0].set(xlabel='x',ylabel='q_x(x)',title='Two trained grids')
    axes[0].legend()
    axes[1].bar(['Damped','Raw scores'],[damped_result['stderr'],raw_result['stderr']],color=['#007c91','#bc5c21'])
    axes[1].set(ylabel='Estimated standard error',title='Same production sample count')
    plt.show()


def diagonal_picture(xedges,yedges):
    visual()
    xx=np.linspace(XMIN,XMAX,301,endpoint=False)
    yy=np.linspace(YMIN,YMAX,301,endpoint=False)
    ix=np.searchsorted(xedges,xx,side='right')-1
    iy=np.searchsorted(yedges,yy,side='right')-1
    qx=1/((len(xedges)-1)*np.diff(xedges)[ix])
    qy=1/((len(yedges)-1)*np.diff(yedges)[iy])
    target=f(xx[:,None],yy[None,:])/exact_integral()
    proposal=qx[:,None]*qy[None,:]
    fig,axes=plt.subplots(1,2,figsize=(9,3.5))
    for ax,array,title in zip(axes,[target,proposal],['Target: two diagonal peaks','Product proposal: four favored regions']):
        im=ax.imshow(array.T,origin='lower',extent=(XMIN,XMAX,YMIN,YMAX),
                     aspect='auto',cmap='viridis',vmin=0,vmax=max(target.max(),proposal.max()))
        ax.set(xlabel='x',ylabel='y',title=title)
    fig.colorbar(im,ax=axes,label='Probability density',shrink=.85)
    plt.show()

## Recall how we measured an average

For Ising, we averaged magnetization over sampled lattices. For the oscillator, we averaged quantities measured on sampled paths.

Here the quantity we will average is a **weight**. Choosing a useful sampling distribution will make those weights fluctuate less.

With a frozen proposal, our draws are independent. The standard error of their mean is $s/\sqrt{N}$.

## An integral and an event distribution

Let $f(x,y)$ be a positive function on a finite region.

$$I=\int f(x,y)\,dx\,dy,\qquad p_{\rm target}(x,y)=\frac{f(x,y)}{I}.$$

In particle physics, $f$ can represent a differential cross section. Its integral gives the total cross section; its normalized shape tells us where events occur.

We will calculate $I$ and generate events with density $p_{\rm target}$.

## Our test function

$$f(x,y)=\sum_{k=1}^{2} A_k\exp\!\left[-\frac12\left(\frac{(x-\mu_{kx})^2}{\sigma_{kx}^2}+\frac{(y-\mu_{ky})^2}{\sigma_{ky}^2}\right)\right].$$

Both peaks are centered at $(2.5,2.5)$ on $0<x,y<5$. The broad peak has $A_1=1$, $\sigma_{1x}=\sigma_{1y}=1$. The narrow peak has $A_2=4$, $\sigma_{2x}=\sigma_{2y}=0.1$.

The analytic finite-square integral lets us check our numerical answer.

## Look at the peak before sampling

The central spike is taller, but it occupies much less area. A uniform grid or uniform random sample needs enough resolution to find it.

In [ ]:
target_picture()

## Height and total contribution

A Gaussian's full-plane integral is $2\pi A\sigma_x\sigma_y$.

The narrow component is four times taller and ten times narrower along each axis. Its full-plane contribution is $4\times0.1\times0.1=0.04$ times the broad component's.

For our finite square, `exact_integral` integrates each component using error functions.

In [ ]:
print(f"Exact finite-square integral: {exact_integral():.8f}")

## Uniform rejection sampling

Draw a point uniformly in the square, then accept it with probability $f(x,y)/f_{\max}$.

The accepted points follow the target distribution. For the centered example, $f_{\max}=5$ and the square's area is 25.

$$P(\mathrm{accept})=\frac{I}{25\times5}\approx0.051.$$

About one proposal in twenty is accepted.

The corrected standard-width Gaussian gives 5.1% uniform acceptance. The earlier notebook implemented broader Gaussians and therefore showed about 9%.

## Compute an integral with uniform draws

A uniform density on this square is $q=1/25$. Therefore

$$I=25\,\mathbb E_{\rm uniform}[f],\qquad \widehat I=\frac1N\sum_{i=1}^{N}25 f(x_i,y_i).$$

The weights are small across much of the square and large near the peak. Their spread determines the statistical error.

In [ ]:
uniform_x = np.array([XMIN, XMAX])
uniform_y = np.array([YMIN, YMAX])
uniform = estimate_integral(uniform_x, uniform_y, 20_000, seed=143)
print(f"Uniform: {uniform['mean']:.4f} ± {uniform['stderr']:.4f}")

## Choose a proposal density

We can draw from any normalized density $q$ that covers the target's support:

$$I=\int q(x,y)\frac{f(x,y)}{q(x,y)}\,dx\,dy
  =\mathbb E_q\!\left[\frac{f}{q}\right].$$

The weight $w=f/q$ corrects for our choice of sampling density. A region sampled more often contributes less weight per point.

## Why matching the shape helps

$$\operatorname{Var}(\widehat I)=\frac1N\left[\int\frac{f^2}{q}\,dx\,dy-I^2\right].$$

For this positive target, the ideal choice $q=f/I$ makes every weight equal to $I$.

VEGAS learns an approximation to that shape. At fixed sample count, a smaller spread of weights gives a smaller uncertainty.

## Equal bin probabilities, unequal widths

Divide the $x$ axis into $B$ intervals. Choose an interval uniformly, then choose $x$ uniformly inside it.

$$q_x(x)=\frac{1}{B\,\Delta x_i}\quad\text{inside interval }i.$$

Every interval has probability $1/B$. Narrower intervals therefore have higher probability density. Moving boundaries toward a peak sends more samples there.

## A change of variables

Think of a uniform random coordinate $u\in[0,1]$. Each interval of width $1/B$ in $u$ maps onto an interval of width $\Delta x_i$ in $x$.

$$\frac{dx}{du}=B\Delta x_i,\qquad q_x=\frac{du}{dx}.$$

The same bin widths can be read as a sampling density or as a Jacobian. Both lead to the weight $f/q$.

## Learn the grid

Start with uniform intervals, draw a training sample, and estimate which intervals contribute to the squared weights. Move the boundaries and repeat.

The squared quantities come from the variance we want to reduce. Damping controls how strongly the grid responds to each noisy training sample.

This run uses 40 intervals per axis, 2,000 evaluations per iteration, and 20 iterations.

In [ ]:
xLow, yLow, _, _ = setup_intervals(seed=142)
training_evaluations = 20 * 2000
print(f"Training evaluations: {training_evaluations:,}")

## Inspect the learned intervals

Look for narrower intervals near the center. Each interval still gets the same share of proposals.

In [ ]:
proposal_picture(xLow, yLow)

## Sample in two dimensions

This implementation learns each coordinate separately:

$$q(x,y)=q_x(x)q_y(y)=\frac{1}{B^2\Delta x_i\Delta y_j}.$$

The importance weight is therefore

$$w=f(x,y)B^2\Delta x_i\Delta y_j.$$

Each coordinate must use the width of its own interval.

## Freeze the grid and measure

After training, use an independent, fixed-size sample. Calculate the mean weight and its standard error.

The analytic answer provides a separate check. The uncertainty below describes production sampling with the trained grid held fixed.

In [ ]:
adaptive = estimate_integral(xLow, yLow, 20_000, seed=144)
print(f"Adaptive: {adaptive['mean']:.4f} ± {adaptive['stderr']:.4f}")
print(f"Exact:    {exact_integral():.6f}")

## Compare the weights

Both averages estimate the same integral. Look at how much the individual weights fluctuate in each sample.

In [ ]:
weight_picture(uniform, adaptive)

## Compare the estimates

Each production estimate uses 20,000 evaluations. The error bars are one estimated standard error. The adaptive result also required 40,000 training evaluations.

In [ ]:
estimate_picture(uniform, adaptive)

## Turn weighted proposals into events

Let $M$ bound the weight everywhere: $M\ge f/q$. Accept a proposal with probability $w/M$.

$$q(x,y)P(\mathrm{accept}\mid x,y)
=q(x,y)\frac{f(x,y)}{M q(x,y)}=\frac{f(x,y)}{M}.$$

The accepted events have the target shape and can all carry equal weight.

## Establish the rejection bound

Within a rectangular grid cell, each Gaussian is largest at the point closest to its center. Sum those component maxima to bound the function in that cell.

Multiply by the cell's constant inverse proposal density, then take the largest bound over all cells.

This construction gives a conservative bound for our positive Gaussian mixture. A largest weight observed in a finite sample would not establish that bound.

## Generate 100,000 events with each method

The adaptive sampler trains on one random stream and generates events on another. Both methods return exactly 100,000 accepted events.

Production runs in batches; the evaluation count includes the whole final batch.

In [ ]:
bf = brute_force(100_000, seed=145)
vg = vegas(100_000, seed=146)
print(f"Returned events: uniform={len(bf['events']):,}, adaptive={len(vg['events']):,}")

## Compare proposals and accepted events

The first and third panels should reproduce the same target shape within sampling fluctuations. The middle panel shows the distribution learned by the adaptive sampler.

In [ ]:
event_picture(bf, vg)

## Count training and production

Compare the total number of function evaluations needed for the same number of accepted events. A trained grid can be reused while the target and domain stay fixed.

The original defaults spent eight million evaluations on training alone. Here the training budget is 40,000.

In [ ]:
cost_picture(bf, vg)

## Compare damping with raw bin scores

The original first exercise removes the damping transformation. We can run that comparison with the same training budget and seed.

Inspect both the learned density and the production uncertainty. One run does not establish which setting works best across targets.

In [ ]:
raw_x, raw_y, _, _ = setup_intervals(alpha_damp=None, seed=142)
raw_result = estimate_integral(raw_x, raw_y, 20_000, seed=144)
damping_picture(xLow, raw_x, adaptive, raw_result)

## Separate the peaks along a diagonal

The original second exercise asks what happens when the Gaussian centers separate. Use centers at (−2, −2) and (2, 2), equal amplitudes, and width 0.5 on the square from −5 to 5.

Each coordinate now has two favored regions. A product proposal combines every favored x region with every favored y region.

In [ ]:
parameter_names = ('A1','A2','MU1X','MU1Y','MU2X','MU2Y',
                   'SIGMA1X','SIGMA1Y','SIGMA2X','SIGMA2Y','XMIN','XMAX','YMIN','YMAX')
baseline_parameters = {name: globals()[name] for name in parameter_names}
diagonal_parameters = dict(A1=1., A2=1., MU1X=-2., MU1Y=-2., MU2X=2., MU2Y=2.,
                           SIGMA1X=.5, SIGMA1Y=.5, SIGMA2X=.5, SIGMA2Y=.5,
                           XMIN=-5., XMAX=5., YMIN=-5., YMAX=5.)

## Correlations limit the product proposal

There are two peaks in the target, but the learned product density also favors the two opposite corners. Those extra regions consume evaluations.

A useful change of coordinates can align a correlated target with the axes. More advanced VEGAS implementations also adapt how they allocate samples across multidimensional regions.

In [ ]:
globals().update(diagonal_parameters)
try:
    diagonal_x, diagonal_y, _, _ = setup_intervals(seed=147)
    diagonal_picture(diagonal_x, diagonal_y)
finally:
    globals().update(baseline_parameters)

## What we can now calculate

Use the average of $f/q$ to estimate an integral, and the spread of those weights to estimate its uncertainty.

Use adaptive intervals to concentrate evaluations where they help. Inspect whether a product of coordinate densities can represent the important structure.

Use a valid bound on $f/q$ to generate equally weighted events, and include training when comparing computational cost.

## References and implementation

G. P. Lepage, “A New Algorithm for Adaptive Multidimensional Integration,” *Journal of Computational Physics* **27**, 192–203 (1978).

G. P. Lepage, “Adaptive multidimensional integration: VEGAS enhanced,” *Journal of Computational Physics* **439**, 110386 (2021).

[How VEGAS works](https://vegas.readthedocs.io/en/latest/background.html)

This notebook implements adaptive product-density sampling for a positive Gaussian mixture. Its rejection bound uses the analytic form of those Gaussians.